# Week 2 Session 2: Baseline Models for Risk Prediction

In Session 1, you engineered 6 features using test-driven development and generated a training dataset of ~580,000 labeled instances. Now you will use that data to answer a fundamental question in any ML project:

> **How well can simple models predict 30-day adverse events for diabetic patients?**

The answer to this question establishes the **performance floor** -- the minimum any future model must beat to justify its complexity.

---

### Why Baselines Before Complex Models?

Baselines serve four purposes that are easy to underestimate:

| Purpose | What It Prevents |
|---------|-----------------|
| **Sanity check** | If a deep learning model scores below a simple rule, something is wrong with the pipeline |
| **Interpretability benchmark** | Simple rules are explainable to clinicians; complex models must justify their opacity |
| **Cost-benefit analysis** | A model that scores 2% better but costs 10x more to maintain may not be worth it |
| **Clinical grounding** | Many hospitals already use rule-based risk scores -- your ML model competes against these existing workflows |

> **Real-world context:** Many hospitals already use simple rules like "HbA1c > 9% triggers care management outreach." Your ML model needs to demonstrate clear improvement over these existing practices to justify adoption.

---

### What You Will Build

| # | Baseline | Approach | Purpose |
|---|----------|----------|---------|
| 1 | Random prediction | Coin flip weighted by class frequency | Absolute floor -- any model must beat this |
| 2 | Majority class | Always predict "low risk" | Exposes why accuracy is misleading with imbalanced data |
| 3 | HbA1c > 9% | Single clinical threshold | Tests whether one strong feature is sufficient |
| 4 | Risk factor counting | Count multiple risk indicators | Tests whether combining features as rules helps |
| 5 | Logistic regression | First real ML model | Tests whether learning weights from data improves on rules |

Each baseline adds one layer of sophistication. If a more complex model cannot beat Baseline 5, it is not adding value.

---

## Setup

We need `scikit-learn` for model training, evaluation metrics, and the patient-level splitting strategy. The training data was generated in Session 1 and saved as a CSV.

## Setting Environment Up for Colab

Mount Google Drive and set the repository path so this notebook can access the EHR data and source code.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

### Project Configuration

This notebook loads paths from a YAML config file instead of hard-coded paths.

**Setup (one-time):**
1. In Colab's left sidebar, click the **Key** icon (Secrets)
2. Add a secret named `PROJECT_CONFIG_PATH`
3. Set the value to your config file path (e.g., `/content/drive/MyDrive/Project/config.yaml`)
4. Toggle "Notebook access" ON

In [ ]:
from google.colab import userdata
import yaml

try:
    config_path = userdata.get('PROJECT_CONFIG_PATH')
except:
    config_path = input("Enter path to your config.yaml: ")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

REPO_PATH = config['repo_path']
print(f"Repository path: {REPO_PATH}")

### Install Source Code as Package

`pip install` installs the local source code as a Python package so you can import directly from it (e.g., `from helpers import get_col`). Re-run this cell after making changes to the source code.

In [ ]:
!pip install {REPO_PATH}/src/ -q

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
import matplotlib.pyplot as plt
import os

print("Libraries loaded")

### Load Pre-Computed Training Data

The training data was generated at the end of Session 1 using `ExtendedPatientProfile.generate_all_instances_session_1()`. Each row represents one (patient, date) pair with 6 features and a binary label.

| Column | Role |
|--------|------|
| `patient_id` | Patient identifier (for grouping, not a feature) |
| `date` | Cutoff date (for reference, not a feature) |
| `days_since_last_hba1c` | Feature: care gap indicator |
| `current_hba1c_level` | Feature: glycemic control |
| `encounters_last_90d` | Feature: utilization intensity |
| `age_at_date` | Feature: age-related risk |
| `current_systolic_bp` | Feature: cardiovascular risk |
| `current_egfr` | Feature: kidney function |
| `will_have_high_risk_event_next_30d` | **Label**: 1 if ED visit, hospitalization, or HbA1c >= 10% within 30 days |

In [ ]:
# Load the training data generated in Session 1
DATA_DIR = os.path.join(REPO_PATH, "data/week_2")
classifier_data = pd.read_csv(os.path.join(DATA_DIR, "classifier_training_data.csv"))

print(f"Training data shape: {classifier_data.shape}")
print(f"Columns: {list(classifier_data.columns)}")
print(f"\nFirst few rows:")
classifier_data.head()

### Understanding the Class Imbalance

Before building any model, we need to understand the label distribution. In clinical event prediction, the vast majority of patient-days are *not* immediately followed by an adverse event. This creates **class imbalance** -- a fundamental challenge that shapes every modeling decision in this session.

**Why it matters:**

| If You Ignore Imbalance... | What Happens |
|---------------------------|-------------|
| Use default model settings | Model learns to predict "low risk" for everything |
| Evaluate with accuracy only | 97% accuracy looks great but means 0% detection of high-risk patients |
| Split data randomly by row | Correlated observations from the same patient leak between train/test |

We will address each of these in this session.

In [ ]:
# Identify the label column
label_col = 'will_have_high_risk_event_next_30d'

# Drop any rows with NaN labels
valid_mask = classifier_data[label_col].notna()
classifier_data = classifier_data[valid_mask].copy()

print(f"Label column: {label_col}")
print(f"\nLabel distribution:")
print(classifier_data[label_col].value_counts().to_string())

n_positive = int(classifier_data[label_col].sum())
n_negative = len(classifier_data) - n_positive
ratio = n_negative / n_positive if n_positive > 0 else float('inf')

print(f"\nPositive rate: {n_positive / len(classifier_data):.2%}")
print(f"Imbalance ratio: {ratio:.0f}:1")
print(f"\nInterpretation: For every high-risk instance, there are ~{ratio:.0f} low-risk instances.")

### Define the Feature Set

We use the **6 features engineered and verified in Session 1**. These cover four distinct aspects of patient risk:

| Feature | Pattern | What It Captures |
|---------|---------|-----------------|
| `days_since_last_hba1c` | Temporal counter | Is the patient being monitored? (care gap detection) |
| `current_hba1c_level` | Clinical value | How well is diabetes being managed? |
| `encounters_last_90d` | Rolling window | Is the patient in a crisis utilization pattern? |
| `age_at_date` | Demographic | Age-related complication risk |
| `current_systolic_bp` | Clinical value | Is hypertension accelerating complications? |
| `current_egfr` | Clinical value | Is kidney function declining? |

In the homework, you will expand this to all 22 features available in `ExtendedPatientProfile.get_daily_features()` and observe whether more features improve baseline performance.

In [ ]:
# Define the 6 Session 1 features
SESSION_1_FEATURES = [
    'days_since_last_hba1c',
    'current_hba1c_level',
    'encounters_last_90d',
    'age_at_date',
    'current_systolic_bp',
    'current_egfr'
]

# Verify all features exist in the data
feature_cols = [col for col in SESSION_1_FEATURES if col in classifier_data.columns]
missing = [col for col in SESSION_1_FEATURES if col not in classifier_data.columns]

print(f"Features found: {len(feature_cols)}/{len(SESSION_1_FEATURES)}")
for col in SESSION_1_FEATURES:
    status = "found" if col in feature_cols else "MISSING"
    print(f"  {col}: {status}")

if missing:
    print(f"\nWARNING: Missing features: {missing}")

### Feature Summary Statistics

Before modeling, we examine the distribution of each feature. This helps identify:

- **Missing values**: Some features may have `NaN` for patients without the relevant observation (e.g., no eGFR reading)
- **Scale differences**: Age ranges 12-96 while encounters_last_90d ranges 0-12 -- logistic regression is sensitive to scale
- **Outliers**: Extreme values that may disproportionately influence model weights

In [ ]:
# Summary statistics
stats = classifier_data[feature_cols].describe().round(2)
print("Feature Statistics:")
print(stats.to_string())

# Missing value analysis
print("\nMissing Values:")
for col in feature_cols:
    n_missing = classifier_data[col].isna().sum()
    pct = n_missing / len(classifier_data) * 100
    print(f"  {col}: {n_missing:,} ({pct:.1f}%)")

**Data quality note:** The minimum `current_hba1c_level` is -0.10, which is clinically impossible (valid HbA1c range: 4.0-15.0%). This is a synthetic data artifact. In production, you would add validation to clip or exclude values outside the valid range. For this session, the impact is minimal since very few instances have this value.

**Missing values:** `current_egfr` is missing for 47.2% of instances -- nearly half. This has significant implications for how the model interprets this feature, as we will see in the feature weights analysis below.

---

## Patient-Level Train/Test Split

This is the most critical data preparation step in this session, and one of the most common sources of leakage in healthcare ML.

### The Problem with Random Splitting

If we randomly split *rows* into train and test, the same patient will have observations in both sets. Since a patient's observations are correlated (same demographics, similar lab trajectories, same disease progression), the model effectively "memorizes" patient patterns from training rows and recognizes them in test rows. This inflates all metrics.

```
Random row-level split (WRONG):
  Patient A, day 1  --> Train
  Patient A, day 8  --> Test    <-- Leaks patient A's pattern!
  Patient A, day 15 --> Train
  Patient B, day 1  --> Test
  Patient B, day 8  --> Train   <-- Leaks patient B's pattern!
```

### The Solution: GroupShuffleSplit

We split by **patient**, not by row. All observations from one patient go entirely into train *or* entirely into test. This simulates the real-world scenario where the model encounters a patient it has never seen before.

```
Patient-level split (CORRECT):
  Patient A, day 1  --> Train
  Patient A, day 8  --> Train   <-- All of Patient A in train
  Patient A, day 15 --> Train
  Patient B, day 1  --> Test
  Patient B, day 8  --> Test    <-- All of Patient B in test
```

We use an 80/20 split: 80% of *patients* for training, 20% for testing.

In [ ]:
# Patient-level split using GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

X = classifier_data[feature_cols].copy()
y = classifier_data[label_col].astype(int).copy()
groups = classifier_data['patient_id'].copy()

# Fill missing feature values with 0 for now
# (We'll examine the impact of this choice later in this session)
X = X.fillna(0)

for train_idx, test_idx in gss.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    train_patients = groups.iloc[train_idx]
    test_patients = groups.iloc[test_idx]

# Verify the split
train_pids = set(train_patients.unique())
test_pids = set(test_patients.unique())
overlap = train_pids & test_pids

print(f"Train: {len(X_train):,} instances from {len(train_pids)} patients")
print(f"Test:  {len(X_test):,} instances from {len(test_pids)} patients")
print(f"\nPatient overlap: {len(overlap)} (must be 0)")

print(f"\nTrain positive rate: {y_train.mean():.2%}")
print(f"Test positive rate:  {y_test.mean():.2%}")

---

## Evaluation Framework

Before building any baselines, we define a consistent evaluation function that every model will use. This ensures fair comparison.

### Which Metrics Matter for Imbalanced Data?

| Metric | Formula | What It Tells You | Reliable with Imbalance? |
|--------|---------|-------------------|-------------------------|
| **Accuracy** | (TP + TN) / Total | Overall correctness | No -- inflated by majority class |
| **Precision** | TP / (TP + FP) | Of patients we flagged, how many were truly high-risk? | Yes |
| **Recall** | TP / (TP + FN) | Of truly high-risk patients, how many did we catch? | Yes |
| **F1 Score** | 2 * P * R / (P + R) | Harmonic mean of precision and recall | Yes |
| **AUC-ROC** | Area under ROC curve | Discrimination ability across all thresholds | Yes (requires probability output) |

For clinical risk prediction, **recall** is often prioritized -- missing a high-risk patient (false negative) is typically more dangerous than a false alarm (false positive). However, very low precision creates alert fatigue, which causes clinicians to ignore the system entirely.

In [ ]:
def evaluate_model(y_true, y_pred, y_prob=None, model_name="Model"):
    """Evaluate a model with standard metrics. Returns dict of metrics."""
    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0)
    }
    if y_prob is not None:
        metrics['auc_roc'] = roc_auc_score(y_true, y_prob)
    return metrics


def print_evaluation(metrics):
    """Display evaluation metrics in a readable format."""
    print(f"\n{'='*55}")
    print(f"  {metrics['model']}")
    print(f"{'='*55}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1']:.4f}")
    if 'auc_roc' in metrics:
        print(f"  AUC-ROC:   {metrics['auc_roc']:.4f}")
    print()


# Storage for all results
all_results = []
print("Evaluation framework ready")

---

## Baseline 1: Random Prediction

**Strategy:** Flip a weighted coin for each instance. The coin's bias matches the training set's class distribution.

**Purpose:** This is the absolute floor. If any model scores at or below this level, it has learned nothing from the data. The random baseline also provides the expected AUC-ROC of 0.5 -- the diagonal on the ROC curve that represents "no discrimination ability."

**Expected behavior:**
- Accuracy will be around ~95% (not the ~97% negative class rate -- the random coin also introduces some misclassification of the majority class)
- Precision and recall will both be approximately equal to the positive rate (~3%)
- AUC-ROC will be 0.5 (no better than chance)

---

<details>
<summary><strong>Hint 1 — Approach</strong> (click to expand)</summary>

Generate random predictions using the training set's positive rate as the coin's bias. The probability score for AUC-ROC should be constant (every instance gets the same probability).
</details>

<details>
<summary><strong>Hint 2 — Key functions</strong> (click to expand)</summary>

`np.random.choice()`, `np.random.seed()`, `np.full()`, `y_train.mean()`
</details>

In [ ]:
# Baseline 1: Random prediction
np.random.seed(42)

# TODO: Calculate the positive rate from the training set
positive_rate = None

# TODO: Generate random predictions using the positive rate as probability
y_pred_random = None

# TODO: Create a constant probability array (every instance gets the same score)
y_prob_random = None

metrics_random = evaluate_model(y_test, y_pred_random, y_prob_random, "Random Baseline")
print_evaluation(metrics_random)
all_results.append(metrics_random)

---

## Baseline 2: Majority Class (Always Predict Low Risk)

**Strategy:** Predict "low risk" (class 0) for every single instance.

**Purpose:** This baseline demonstrates exactly why **accuracy is a misleading metric** for imbalanced data. With ~97% of instances being low-risk, predicting "low risk" for everyone achieves ~97% accuracy -- a number that looks impressive but is clinically useless because it catches zero high-risk patients.

**Expected behavior:**
- Very high accuracy (equal to negative class rate)
- Precision = 0 (never predicts positive)
- Recall = 0 (catches no high-risk patients)
- F1 = 0 (useless for the task we care about)

> **Key lesson:** Any time you see accuracy > 95% on an imbalanced dataset, check precision and recall immediately. The model might be doing nothing useful.

---

<details>
<summary><strong>Hint 1 — Approach</strong> (click to expand)</summary>

Find which class appears most often in the training set, then predict that class for every test instance.
</details>

<details>
<summary><strong>Hint 2 — Key functions</strong> (click to expand)</summary>

`y_train.mode()`, `np.full()`
</details>

In [ ]:
# Baseline 2: Majority class — always predict the most common label

# TODO: Find the majority class from the training labels
majority_class = None

# TODO: Create an array of predictions (all the same value)
y_pred_majority = None

metrics_majority = evaluate_model(y_test, y_pred_majority, model_name="Majority Class")
print_evaluation(metrics_majority)
all_results.append(metrics_majority)

print(f"This model achieves {metrics_majority['accuracy']:.1%} accuracy")
print(f"but catches {metrics_majority['recall']:.0%} of high-risk patients.")
print(f"\nThis is why accuracy alone is useless for imbalanced data.")

---

## Baseline 3: Single-Feature Clinical Threshold (HbA1c > 9%)

**Strategy:** Use a single clinical rule: if the patient's most recent HbA1c exceeds 9%, predict high risk.

**Purpose:** This tests whether one strong feature, applied with a clinically-grounded threshold, can outperform the trivial baselines. The 9% threshold comes directly from ADA guidelines, which classify HbA1c >= 9% as "poor glycemic control requiring intervention."

**Clinical context:** This rule is already used in many healthcare systems. Care management programs often auto-enroll patients with HbA1c > 9% into diabetes education, nurse follow-up, or medication review. If our ML model cannot beat this simple rule, it adds no value over existing clinical practice.

### How Missing Values Are Handled

Patients without a prior HbA1c reading get `NaN`, which we filled with 0 during data preparation. Since 0 < 9, these patients are predicted as low risk. This is a **conservative** assumption -- if we have no data about a patient's glycemic control, we do not flag them.

### Threshold Sensitivity

The 9% threshold is one reasonable choice, but different thresholds trade off precision and recall differently:

| Threshold | What It Means | Expected Effect |
|-----------|-------------- |-----------------|
| 7.0% | ADA target for most adults | High recall (catches many), low precision (many false alarms) |
| 8.0% | Relaxed target for elderly/comorbid | Moderate balance |
| 9.0% | ADA "poor control" threshold | Our primary baseline |
| 10.0% | Very poor control (one of our label triggers) | High precision (few false alarms), low recall (misses many) |

---

<details>
<summary><strong>Hint 1 — Approach</strong> (click to expand)</summary>

Apply a single threshold to one feature column. Instances where HbA1c exceeds 9% are predicted as high risk. For the probability score, normalize HbA1c to a 0-1 range.
</details>

<details>
<summary><strong>Hint 2 — Key functions</strong> (click to expand)</summary>

`(series > threshold).astype(int)`, `np.clip()`, `confusion_matrix()`
</details>

In [ ]:
# Baseline 3: Clinical threshold — HbA1c > 9% = high risk
THRESHOLD_HBA1C = 9.0
hba1c_col = 'current_hba1c_level'

# TODO: Get HbA1c values from test set (fill NaN with 0)
hba1c_values = None

# TODO: Create binary predictions (1 if HbA1c > threshold, 0 otherwise)
y_pred_hba1c = None

# TODO: Create a probability score by normalizing HbA1c to [0, 1]
# (hint: clip (hba1c - 6) / 6 to the range [0, 1])
y_prob_hba1c = None

metrics_hba1c = evaluate_model(y_test, y_pred_hba1c, y_prob_hba1c, "HbA1c > 9%")
print_evaluation(metrics_hba1c)
all_results.append(metrics_hba1c)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_hba1c)
print(f"Confusion Matrix:")
print(f"                    Predicted")
print(f"                    Low     High")
print(f"  Actual Low:    {cm[0,0]:>7,}  {cm[0,1]:>7,}")
print(f"  Actual High:   {cm[1,0]:>7,}  {cm[1,1]:>7,}")

### Threshold Sensitivity Analysis

We test multiple HbA1c thresholds to see how the precision-recall tradeoff shifts. Lower thresholds catch more high-risk patients (higher recall) but also flag more false positives (lower precision).

In [ ]:
# Sweep HbA1c thresholds
thresholds = [7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0]

print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 48)

for thresh in thresholds:
    y_pred = (X_test[hba1c_col].fillna(0) > thresh).astype(int)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    marker = " <-- selected" if thresh == 9.0 else ""
    print(f"  {thresh:<10.1f} {prec:<12.3f} {rec:<12.3f} {f1:<12.3f}{marker}")

---

## Baseline 4: Risk Factor Counting

**Strategy:** Count how many clinical risk factors are present, then predict high risk if the count exceeds a threshold.

**Purpose:** This tests whether combining *multiple* features as binary indicators outperforms a single-feature threshold. It mirrors how clinicians think: "This patient has elevated HbA1c *and* hasn't been seen in 6 months *and* has declining kidney function -- that is three red flags."

### Risk Factors (Using Session 1 Features)

Each factor is a clinically-motivated binary indicator:

| # | Risk Factor | Threshold | Clinical Rationale |
|---|------------|-----------|-------------------|
| 1 | HbA1c > 8% | Poor glycemic control | ADA: intensify treatment above this level |
| 2 | Care gap > 180 days | Long time since HbA1c test | ADA: test every 3-6 months; >6 months is a gap |
| 3 | Age > 65 | Elderly patient | Higher complication rates, polypharmacy risk |
| 4 | Systolic BP > 140 | Hypertension | Accelerates nephropathy, retinopathy in diabetics |
| 5 | eGFR < 60 | Kidney disease (CKD stage 3+) | KDIGO: refer to nephrology, adjust medications |

**Scoring:** Each present risk factor adds 1 point, giving a score from 0 to 5. We then test different thresholds to determine how many risk factors should trigger a "high risk" prediction.

---

<details>
<summary><strong>Hint 1 — Approach</strong> (click to expand)</summary>

For each row, check 5 binary conditions and count how many are True. Be careful with eGFR: a value of 0 means the test was never ordered (due to fillna), not that the patient has zero kidney function.
</details>

<details>
<summary><strong>Hint 2 — Key functions</strong> (click to expand)</summary>

`row.get()`, comparison operators (note: features have fillna(0) applied, so check `> 0` for eGFR before comparing `< 60`)
</details>

In [ ]:
def count_risk_factors(row):
    """
    Count clinical risk factors present for a patient instance.
    Each factor is a binary indicator based on Session 1 features.
    Returns an integer count from 0 to 5.

    Risk factors:
    1. current_hba1c_level > 8     (poor glycemic control)
    2. days_since_last_hba1c > 180  (care gap)
    3. age_at_date > 65             (elderly patient)
    4. current_systolic_bp > 140    (hypertension)
    5. current_egfr < 60            (kidney disease — but skip if eGFR is 0, which means no test)
    """
    count = 0

    # TODO: Check each of the 5 risk factors and increment count

    return count

print("count_risk_factors function defined")

### Apply Risk Factor Counting

We compute risk factor counts for all test instances and examine the distribution. This tells us what fraction of patients have 0, 1, 2, ... risk factors.

In [ ]:
# Apply risk factor counting to the test set
risk_counts = X_test.apply(count_risk_factors, axis=1)

print("Risk factor count distribution:")
dist = risk_counts.value_counts().sort_index()
for count_val, n in dist.items():
    pct = n / len(risk_counts) * 100
    print(f"  {count_val} risk factors: {n:>8,} instances ({pct:.1f}%)")

### Evaluate at Threshold >= 2

We start with the threshold of **2 or more risk factors** -- a common clinical heuristic where the presence of multiple concurrent risk factors is considered more concerning than any single factor alone.

In [ ]:
# Risk factor baseline: high risk if >= 2 risk factors
RISK_THRESHOLD = 2

y_pred_risk = (risk_counts >= RISK_THRESHOLD).astype(int)
y_prob_risk = risk_counts / 5  # Normalize score to [0, 1]

metrics_risk = evaluate_model(y_test, y_pred_risk, y_prob_risk,
                              f"Risk Factors >= {RISK_THRESHOLD}")
print_evaluation(metrics_risk)
all_results.append(metrics_risk)

### Threshold Sensitivity Analysis

Different thresholds produce different precision-recall tradeoffs:

- **>= 1**: Flag anyone with *any* risk factor -- high recall but many false alarms
- **>= 3**: Only flag patients with *several* risk factors -- fewer false alarms but misses more cases
- **>= 4+**: Very selective -- likely misses most high-risk patients

In [ ]:
# Test different risk factor thresholds
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 48)

for thresh in [1, 2, 3, 4]:
    y_pred = (risk_counts >= thresh).astype(int)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    marker = " <-- selected" if thresh == RISK_THRESHOLD else ""
    print(f"  >= {thresh:<8} {prec:<12.3f} {rec:<12.3f} {f1:<12.3f}{marker}")

---

## Baseline 5: Logistic Regression

**Strategy:** Train a logistic regression model on all 6 features with `class_weight='balanced'` to handle class imbalance.

**Purpose:** This is the first model that *learns* from the data rather than using hardcoded rules. It serves two purposes:
1. **Upper bound for simple models**: If logistic regression cannot beat the clinical rules, the features may lack predictive power
2. **Lower bound for complex models**: Any neural network, gradient boosting, or ensemble model must beat this to justify its complexity

### Why `class_weight='balanced'`?

Without class weighting, logistic regression will converge to the majority-class solution (predict 0 for everything) because that minimizes overall loss. The `balanced` setting automatically adjusts the loss function:

```
weight_positive = n_total / (2 * n_positive)   ~= 18x
weight_negative = n_total / (2 * n_negative)   ~= 0.5x
```

This forces the model to pay ~35x more attention to misclassifying a positive instance than a negative one, counteracting the imbalance.

### Why `StandardScaler`?

Our features are on very different scales: `age_at_date` ranges 12-96, `encounters_last_90d` ranges 0-12, and `current_systolic_bp` ranges 100-200. Without standardization:
- Gradient-based optimization may converge slowly for features on large scales
- Coefficient magnitudes are not directly comparable across features

`StandardScaler` transforms each feature to have mean 0 and standard deviation 1, so each coefficient represents the effect of a one-standard-deviation change. This makes coefficient magnitudes directly comparable and improves convergence.

### What Logistic Regression Learns

Unlike rule-based baselines, logistic regression learns **optimal weights** for each feature from the training data:

```
P(high_risk) = sigmoid(w1*hba1c + w2*days_since + w3*encounters + w4*age + w5*bp + w6*egfr + bias)
```

The learned weights tell us which features the model considers most predictive -- a useful interpretability check.

In [ ]:
# Prepare clean data for logistic regression
X_train_clean = X_train.fillna(0)
X_test_clean = X_test.fillna(0)

print(f"Training set: {len(X_train_clean):,} instances, {len(feature_cols)} features")
print(f"Test set:     {len(X_test_clean):,} instances")
print(f"\nPositive rate in training: {y_train.mean():.2%}")

### Train and Evaluate

Logistic regression produces two types of output:
- **Hard predictions** (`predict`): Binary 0/1 labels for confusion matrix, precision, recall
- **Probability scores** (`predict_proba`): Continuous [0, 1] scores for AUC-ROC and threshold tuning

The probability output is especially valuable: in production, clinicians can set different thresholds depending on available resources. A threshold of 0.3 catches more patients (higher recall); a threshold of 0.7 focuses on the highest-risk patients (higher precision).

---

<details>
<summary><strong>Hint 1 — Approach</strong> (click to expand)</summary>

Three steps: (1) standardize features with `StandardScaler` (fit on train, transform both), (2) train `LogisticRegression` with `class_weight='balanced'`, (3) generate both hard predictions and probability scores.
</details>

<details>
<summary><strong>Hint 2 — Key functions</strong> (click to expand)</summary>

`StandardScaler().fit_transform()` / `.transform()`, `LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)`, `.predict()`, `.predict_proba()[:, 1]`
</details>

In [ ]:
# Baseline 5: Logistic Regression with class balancing

# TODO: Standardize features using X_train_clean/X_test_clean from above
scaler = None
X_train_scaled = None
X_test_scaled = None

# TODO: Train logistic regression with class_weight='balanced'
lr_model = None

# TODO: Generate hard predictions and probability scores
y_pred_lr = None
y_prob_lr = None

metrics_lr = evaluate_model(y_test, y_pred_lr, y_prob_lr, "Logistic Regression")
print_evaluation(metrics_lr)
all_results.append(metrics_lr)

### Detailed Classification Report

The classification report shows per-class metrics. For our task, focus on the **High Risk** row -- that is the class we are trying to predict. The **support** column shows the number of instances in each class, confirming the imbalance.

In [ ]:
# Detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Low Risk', 'High Risk']))

### Feature Weights (Learned Coefficients)

One advantage of logistic regression is full interpretability. The learned coefficients tell us the direction and relative importance of each feature:

- **Positive coefficient**: Higher feature value --> higher risk
- **Negative coefficient**: Higher feature value --> lower risk

Since we applied `StandardScaler`, each coefficient represents the effect of a **one-standard-deviation change** in that feature. Larger absolute values indicate stronger influence on the prediction. This lets us directly compare which features matter most.

In [ ]:
# Display learned feature weights
print("Logistic Regression Feature Weights:")
print(f"{'Feature':<30} {'Coefficient':<12} {'Direction':<15}")
print("-" * 57)

for feature, coef in sorted(zip(feature_cols, lr_model.coef_[0]),
                              key=lambda x: abs(x[1]), reverse=True):
    direction = "Higher -> MORE risk" if coef > 0 else "Higher -> LESS risk"
    print(f"  {feature:<28} {coef:<+12.4f} {direction}")

print(f"\n  {'Intercept (bias)':<28} {lr_model.intercept_[0]:<+12.4f}")

### Why Do Some Coefficients Seem Clinically Backwards?

Look at the coefficients above: the model says higher systolic BP leads to *less* risk, and higher eGFR leads to *more* risk. Both contradict clinical knowledge -- higher BP is a well-established cardiovascular risk factor, and higher eGFR means better kidney function.

These reversals have different root causes:

**1. Systolic BP: an imputation artifact**

When we filled missing values with `fillna(0)`, we set ~20K instances (3.5% missing) to BP = 0. Patients without a recorded blood pressure tend to have fewer encounters, which correlates with lower risk. The model learns "BP = 0 means low risk," pulling the coefficient negative. **Median imputation fixes this** -- replacing 0 with a typical BP value removes the confound and flips the coefficient to the clinically correct positive direction.

**2. eGFR: a synthetic data artifact**

The eGFR situation is more subtle. With 47.2% of values missing, you might expect `fillna(0)` to be the culprit. But median imputation does *not* flip the eGFR coefficient. The positive direction is baked into the data itself:

| Pattern | Positive Rate | Explanation |
|---------|:------------:|-------------|
| eGFR missing | 2.1% | No test ordered -- likely healthier patients |
| eGFR present | 3.5% | Test ordered -- clinical concern exists |
| eGFR in Q4 (highest values) | 3.6% | More comprehensive workups in sicker patients |

In Synthea's synthetic data, ~75% of non-missing eGFR values cluster at exactly 60.0, with higher values appearing in patients who receive more frequent monitoring. The model correctly learns this pattern -- but it reflects data generation mechanics, not real clinical relationships.

**Takeaway:** Counterintuitive coefficients can have different explanations. Some are fixable with better preprocessing (BP), while others reflect data-level confounds that require domain-aware feature engineering (eGFR). Let's see this in action:

In [ ]:
# Compare imputation strategies: fillna(0) vs fillna(median)
from sklearn.preprocessing import StandardScaler

# Re-extract raw features (before fillna) using the same train/test indices
X_raw_train = classifier_data[feature_cols].iloc[train_idx].copy()
X_raw_test = classifier_data[feature_cols].iloc[test_idx].copy()

print("NaN counts in raw training data:")
for col in feature_cols:
    n_nan = X_raw_train[col].isna().sum()
    pct = n_nan / len(X_raw_train) * 100
    print(f"  {col}: {n_nan:,} ({pct:.1f}%)")

# Strategy 1: fillna(0)
X_train_zero = X_raw_train.fillna(0)
X_test_zero = X_raw_test.fillna(0)

# Strategy 2: fillna(median) — fill with training set median
train_medians = X_raw_train.median()
X_train_median = X_raw_train.fillna(train_medians)
X_test_median = X_raw_test.fillna(train_medians)

print(f"\nTraining set medians (computed from non-NaN values only):")
for col in feature_cols:
    print(f"  {col}: {train_medians[col]:.2f}")

# Standardize both so coefficient magnitudes are comparable
scaler_zero = StandardScaler()
scaler_median = StandardScaler()

X_train_zero_scaled = scaler_zero.fit_transform(X_train_zero)
X_test_zero_scaled = scaler_zero.transform(X_test_zero)

X_train_median_scaled = scaler_median.fit_transform(X_train_median)
X_test_median_scaled = scaler_median.transform(X_test_median)

# Train both models
lr_zero = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_zero.fit(X_train_zero_scaled, y_train)

lr_median = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_median.fit(X_train_median_scaled, y_train)

# Compare standardized coefficients
print("\nStandardized Coefficients: fillna(0) vs fillna(median)")
print("=" * 75)
print(f"{'Feature':<30} {'Zero-Fill':<15} {'Median-Fill':<15} {'Sign Change?'}")
print("-" * 75)

for i, feature in enumerate(feature_cols):
    coef_zero = lr_zero.coef_[0][i]
    coef_median = lr_median.coef_[0][i]
    sign_change = "  <-- YES" if (coef_zero > 0) != (coef_median > 0) else ""
    print(f"  {feature:<28} {coef_zero:>+10.4f}     {coef_median:>+10.4f}{sign_change}")

# Compare predictive performance
auc_zero = roc_auc_score(y_test, lr_zero.predict_proba(X_test_zero_scaled)[:, 1])
auc_median = roc_auc_score(y_test, lr_median.predict_proba(X_test_median_scaled)[:, 1])
print(f"\nROC-AUC with zero-fill:   {auc_zero:.4f}")
print(f"ROC-AUC with median-fill: {auc_median:.4f}")

### Imputation Matters

The results confirm the two distinct mechanisms:

1. **`current_systolic_bp` flipped sign.** Under `fillna(0)` the coefficient was negative (counterintuitive); under median imputation it became positive (clinically correct). This is the imputation confound in action -- `fillna(0)` created a spurious "BP = 0 means healthy" signal that median imputation removes.

2. **`current_egfr` stayed positive in both.** Changing the imputation strategy did not fix this coefficient because the positive direction is a property of the synthetic data itself, not just a `fillna(0)` artifact. This is a deeper confound that would require feature engineering (e.g., a binary `has_egfr_test` indicator) rather than just better imputation.

3. **Standardized coefficients are now comparable.** After standardization, the magnitudes directly reflect each feature's influence per standard deviation. `days_since_last_hba1c` and `current_hba1c_level` dominate, which is clinically sensible.

**Takeaway:** Imputation is not a neutral preprocessing step -- it encodes assumptions about what missing data means. In healthcare:
- `fillna(0)` for a lab test creates a confound between "test not ordered" and "extreme low value"
- `fillna(median)` says "assume a typical result for the population"
- Neither is always correct -- the right choice depends on *why* the data is missing

For this session, we continue with `fillna(0)` for the remaining analysis to keep things simple, but keep this limitation in mind when interpreting model coefficients.

---

## Comparison: All Baselines

Now we compare all 5 baselines side by side. The key metrics to focus on are **F1** (balances precision and recall) and **AUC-ROC** (measures discrimination ability across all thresholds).

Remember:
- **Accuracy** is inflated by class imbalance -- do not use it to compare models
- **Precision** alone can be gamed by being very selective (predict positive rarely)
- **Recall** alone can be gamed by predicting positive always
- **F1** balances both, and **AUC-ROC** gives a threshold-independent view

In [ ]:
# Create comparison table
results_df = pd.DataFrame(all_results).set_index('model')

print("Baseline Comparison")
print("=" * 75)
print(results_df.round(4).to_string())

### Visual Comparison

Two complementary views of model performance:

**Left -- Precision vs Recall scatter:** The ideal model lives in the upper-right corner (high precision AND high recall). Most real models trade one for the other. This plot shows where each baseline falls on that tradeoff.

**Right -- F1 Score bar chart:** A single-number summary that balances precision and recall. Higher bars indicate better overall performance for the minority class prediction task.

In [ ]:
# Visualize baseline comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Precision vs Recall
ax1 = axes[0]
for idx, row in results_df.iterrows():
    ax1.scatter(row['recall'], row['precision'], s=120, zorder=5, label=idx)
ax1.set_xlabel('Recall (sensitivity)', fontsize=11)
ax1.set_ylabel('Precision (positive predictive value)', fontsize=11)
ax1.set_title('Precision vs Recall Tradeoff', fontsize=12)
ax1.set_xlim(-0.05, 1.05)
ax1.set_ylim(-0.05, max(results_df['precision'].max() * 1.3, 0.3))
ax1.grid(True, alpha=0.3)
ax1.axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.5, label='Random precision')
ax1.legend(fontsize=9, loc='upper right')

# Right: F1 Score bars
ax2 = axes[1]
colors = plt.cm.viridis(np.linspace(0.1, 0.85, len(results_df)))
bars = ax2.barh(results_df.index, results_df['f1'], color=colors, edgecolor='white')
ax2.set_xlabel('F1 Score', fontsize=11)
ax2.set_title('F1 Score Comparison', fontsize=12)
for bar, val in zip(bars, results_df['f1']):
    if val > 0:
        ax2.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### Interpreting the Results

Several patterns emerge from the comparison:

**1. Accuracy is meaningless here.** The majority class baseline achieves the *highest* accuracy of any model, yet it catches zero high-risk patients. Any interpretation of these results that relies on accuracy is wrong.

**2. A single clinical rule (HbA1c > 9%) achieves the highest F1 score.** Despite using only one feature, it outperforms logistic regression on F1 because it achieves much higher precision. However, logistic regression achieves significantly higher AUC-ROC and recall, meaning it is better at *ranking* patients by risk and catches far more true positives overall. The tradeoff: the clinical rule is more precise when it fires, but the ML model catches more high-risk patients. This illustrates why no single metric tells the full story.

**3. Risk factor counting underperforms.** Combining features as binary indicators (present/absent) throws away valuable information. A patient with HbA1c = 8.1% and one with HbA1c = 14% both get "1 point" for the HbA1c risk factor, even though their risk levels are drastically different. Logistic regression preserves this continuous information.

**4. Logistic regression learns from data.** With standardized features, the coefficients reveal which features have the strongest influence on predictions. However, some coefficient signs are counterintuitive due to `fillna(0)` artifacts (see the discussion above on imputation confounds).

**5. The positive class is hard to predict.** Even the best baseline has modest F1 scores. This is expected -- predicting rare events is inherently difficult. Session 3 will introduce more sophisticated evaluation metrics (PR-AUC, precision at fixed recall) designed for this scenario.

---

## Summary

### What You Accomplished

1. **Loaded and explored** the training data from Session 1 -- 6 features, ~580K instances, ~35:1 class imbalance

2. **Implemented patient-level data splitting** using `GroupShuffleSplit` to prevent leakage from correlated observations of the same patient

3. **Built and evaluated 5 baseline models** of increasing sophistication:

| Baseline | Key Insight |
|----------|------------|
| Random | AUC-ROC = 0.5 -- the floor for any useful model |
| Majority Class | 97%+ accuracy but 0% recall -- proves accuracy is misleading |
| HbA1c > 9% | A single clinical rule can be surprisingly effective (highest F1) |
| Risk Factor Counting | Binary indicators lose information compared to continuous features |
| Logistic Regression | Learning weights from data achieves the best ranking ability (highest AUC-ROC) |

4. **Established the performance benchmarks** that any more complex model must beat

### Key Takeaways

- **Never report accuracy alone** on imbalanced data. Always include precision, recall, F1, and AUC-ROC.
- **Patient-level splitting** is non-negotiable for correlated longitudinal data. Row-level random splitting inflates all metrics.
- **Simple baselines set the bar.** If a complex model cannot beat logistic regression on AUC-ROC, it is not adding value -- it is adding complexity.
- **Class imbalance requires active handling.** Use `class_weight='balanced'`, adjusted thresholds, or specialized metrics.
- **No single metric tells the whole story.** HbA1c > 9% wins on F1, logistic regression wins on AUC-ROC and recall. The right choice depends on whether you prioritize precision or coverage.

---

## Next Steps

In **Session 3: Evaluation Metrics**, you will:
- Learn why **PR-AUC** (Precision-Recall Area Under Curve) is more informative than ROC-AUC for imbalanced data
- Calculate **precision at fixed recall levels** (e.g., "What precision can we achieve while catching 80% of high-risk patients?")
- Perform **cost-benefit analysis** to translate model performance into clinical and financial impact